#### EXTRACT `LOCATIONS` FROM VOLUME
- /Volumes/oracle_hrms/landing/operational/locations/

In [0]:
from pyspark.sql.functions import *

locations_df = (
    spark.read.format('csv')
    .option('header', 'true')
    .load('/Volumes/oracle_hrms/landing/operational/locations/*')
)
from pyspark.sql.functions import input_file_name, input_file_block_start, input_file_block_length

trans_locations_df = (
    locations_df
    .selectExpr(
        "LOCATION_ID as location_id",
        "LOCATION_NAME as location_name",
        "LOAD_TS as load_ts",
        "_metadata.file_path as file_path",
        "_metadata.file_name as file_name",
        "_metadata.file_modification_time as load_timestamp"
    )
)


trans_locations_df.display()

In [0]:
trans_locations_df.createOrReplaceTempView('locations_temp_vw')

#### CAPTURE INCREMENTAL LOAD & OBSERVABILITY METRICES
- `ORACLE_HRMS.BRONZE.LOCATIONS`
- `ORACLE_HRMS.AUDIT.AUDIT_LOG`

In [0]:
from pyspark.sql import Row
from datetime import datetime
import uuid
import sys
import traceback
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType

# =====================================================================
# 1. INITIALIZE METADATA
# =====================================================================
load_start_time = datetime.now()
pipeline_name = "01.IngestLocationsIncremental"

status = "SUCCESS"
message = "Incremental load completed"
record_count = 0
load_executed = False

try:
    # =====================================================================
    # 2. DETERMINE LATEST TIMESTAMP
    # =====================================================================
    max_ts_row = spark.sql("""
        SELECT MAX(load_timestamp) AS max_load_ts
        FROM oracle_hrms.bronze.locations
    """).collect()[0]

    max_ts = max_ts_row["max_load_ts"]

    # =====================================================================
    # 3. FILTER SOURCE DATA
    # =====================================================================
    if max_ts is None:
        # First load: take all records
        new_records_df = spark.sql("SELECT * FROM locations_temp_vw")
    else:
        # Incremental load: only records newer than max_ts
        new_records_df = spark.sql(f"""
            SELECT *
            FROM locations_temp_vw
            WHERE load_timestamp > TIMESTAMP('{max_ts}')
        """)

    record_count = new_records_df.count()

    # =====================================================================
    # 4. APPLY MERGE IF NEW RECORDS EXIST
    # =====================================================================
    if record_count > 0:
        spark.sql("""
            MERGE INTO oracle_hrms.bronze.locations tgt
            USING locations_temp_vw src
            ON (tgt.location_id = src.location_id)
            WHEN MATCHED AND src.load_timestamp > tgt.load_timestamp THEN 
              UPDATE SET
                tgt.location_name   = src.location_name,
                tgt.load_ts         = src.load_ts,
                tgt.file_path       = src.file_path,
                tgt.file_name       = src.file_name,
                tgt.load_timestamp  = src.load_timestamp
            WHEN NOT MATCHED THEN
              INSERT (location_id, location_name, load_ts, file_path, file_name, load_timestamp)
              VALUES (src.location_id, src.location_name, src.load_ts, src.file_path, src.file_name, src.load_timestamp)
        """)
        load_executed = True
        print(f"Incremental MERGE applied. Rows affected: {record_count}")
    else:
        status = "SKIPPED"
        message = "No new records found. Incremental load skipped."
        print(message)

except Exception as e:
    # =====================================================================
    # 5. EXCEPTION HANDLING
    # =====================================================================
    status = "FAILED"
    exc_type, exc_value, exc_tb = sys.exc_info()
    error_details = ''.join(traceback.format_exception(exc_type, exc_value, exc_tb))
    message = f"Pipeline failed! Error: {error_details}"
    record_count = -1

finally:
    # =====================================================================
    # 6. AUDIT LOGGING
    # =====================================================================
    load_end_time = datetime.now()
    current_date = load_end_time.date()

    # Only log if load executed OR failed
    if load_executed or status == "FAILED":
        try:
            max_run_df = spark.table("ORACLE_HRMS.AUDIT.AUDIT_LOGS") \
                .filter(
                    (F.to_date("event_time") == F.lit(current_date)) &
                    (F.col("pipeline_name") == F.lit(pipeline_name))
                ) \
                .select(F.max(F.col("run_id").cast("int")).alias("max_id"))
            
            max_id_row = max_run_df.collect()[0]
            next_run_int = (max_id_row["max_id"] + 1) if max_id_row["max_id"] is not None else 1
        except Exception:
            next_run_int = 1

        run_id_str = f"{next_run_int:02d}"

        try:
            context = dbutils.notebook.getContext()
            notebook_path = context.notebookPath().get()
            user_name = context.tags().apply("user")
        except Exception:
            notebook_path = "Unknown/Local"
            user_name = "System"

        log_entry = Row(
            log_id=str(uuid.uuid4()),
            run_id=run_id_str,
            event_time=load_end_time,
            event_type="INCREMENTAL LOAD",
            source_table="/Volumes/oracle_hrms/landing/operational/locations/",
            target_table="ORACLE_HRMS.BRONZE.LOCATIONS",
            record_count=record_count,
            status=status,
            message=message,
            user_name=user_name,
            notebook_path=notebook_path,
            pipeline_name=pipeline_name,
            load_start_time=load_start_time,
            load_end_time=load_end_time
        )

        log_schema = StructType([
            StructField("log_id", StringType(), True),
            StructField("run_id", StringType(), True),
            StructField("event_time", TimestampType(), True),
            StructField("event_type", StringType(), True),
            StructField("source_table", StringType(), True),
            StructField("target_table", StringType(), True),
            StructField("record_count", LongType(), True),
            StructField("status", StringType(), True),
            StructField("message", StringType(), True),
            StructField("user_name", StringType(), True),
            StructField("notebook_path", StringType(), True),
            StructField("pipeline_name", StringType(), True),
            StructField("load_start_time", TimestampType(), True),
            StructField("load_end_time", TimestampType(), True)
        ])

        log_entry_df = spark.createDataFrame([log_entry], schema=log_schema)
        log_entry_df.write.format("delta").mode("append").saveAsTable("ORACLE_HRMS.AUDIT.AUDIT_LOGS")
        print(f"[AUDIT LOGGED] Status: {status} | Run ID: {run_id_str} | Count: {record_count}")

        if status == "FAILED":
            raise RuntimeError(message)


In [0]:
%sql
SELECT
COUNT(*) AS cnt,
file_name
FROM oracle_hrms.bronze.locations
GROUP BY file_name;

In [0]:
%sql
DESCRIBE TABLE EXTENDED oracle_hrms.bronze.locations

In [0]:
dbutils.notebook.exit('LOCATIONS LOADED SUCCESSFULLY')

In [0]:
%sql
-- VALIDATE THE QUERY RESULTS FROM `ORACLE_HRMS.BRONZE.LOCATIONS`
SELECT
  *
FROM ORACLE_HRMS.BRONZE.LOCATIONS
ORDER BY 1;

In [0]:
%sql
-- VALIDATE `ORACLE_HRMS.AUDIT.AUDIT_LOGS`
SELECT
*
FROM
ORACLE_HRMS.AUDIT.AUDIT_LOGS;